In [7]:
import requests
import json
import time

OLLAMA_MODEL = "qwen3.5:4b"

def call_llm(prompt, model=OLLAMA_MODEL, temperature=0.7, max_tokens=300):
    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": model,
                "prompt": prompt,
                "stream": False,
                "think": False,
                "options": {"temperature": temperature, "num_predict": max_tokens}
            },
            timeout=60
        )
        return response.json().get("response", "").strip()
    except Exception as e:
        return f"LLM call failed: {e}"

print("Ollama LLM helper loaded")

Ollama LLM helper loaded


In [8]:
with open("../outputs/skills.json", "r", encoding="utf-8") as f:
    skills_data = json.load(f)

with open("../outputs/skill_gap.json", "r", encoding="utf-8") as f:
    gap_data = json.load(f)

with open("../outputs/resume_text.txt", "r", encoding="utf-8") as f:
    resume_text = f.read()

technical_skills = skills_data.get("llm_based", {}).get("technical_skills", skills_data["regex_based"]["skills"]["technical"])
target_role = gap_data["target_role"]

print(f"Target role: {target_role}")
print(f"Technical skills: {technical_skills}")

Target role: AI Engineer
Technical skills: ['Python', 'Java', 'C', 'C++', 'SQL', 'HTML', 'CSS', 'JavaScript', 'NumPy', 'Pandas', 'TensorFlow', 'PyTorch', 'Flask']


In [9]:
TECHNICAL_QUESTIONS = {
    "python": ["Explain the difference between a list and a tuple in Python.", "What are Python decorators and when would you use them?"],
    "machine learning": ["Explain the bias-variance tradeoff.", "What is overfitting and how do you prevent it?"],
    "tensorflow": ["What is the difference between TensorFlow and PyTorch?", "Explain how backpropagation works in a neural network."],
    "pytorch": ["How do you handle GPU memory management in PyTorch?", "Explain the difference between torch.nn.Module and torch.nn.functional."],
    "sql": ["What is the difference between INNER JOIN and LEFT JOIN?", "How would you optimize a slow SQL query?"],
    "flask": ["How does Flask handle routing?", "What is the difference between Flask and FastAPI?"],
    "data science": ["Walk me through your approach to handling missing data in a dataset.", "What's the difference between correlation and causation?"]
}

CODING_QUESTIONS = [
    "Write a function to check if a string is a palindrome.",
    "Given an array of integers, find two numbers that add up to a target sum.",
    "Implement a function to reverse a linked list.",
    "Write a function to find the longest substring without repeating characters."
]

HR_QUESTIONS = [
    "Tell me about yourself.",
    "Why do you want to work in this role?",
    "Where do you see yourself in 5 years?",
    "What is your biggest strength and weakness?",
    "Why should we hire you over other candidates?"
]

BEHAVIORAL_QUESTIONS = [
    "Tell me about a time you faced a challenging project and how you handled it.",
    "Describe a situation where you had to work with a difficult team member.",
    "Give an example of a time you had to learn a new skill quickly.",
    "Tell me about a time you failed and what you learned from it."
]

def generate_interview_questions(technical_skills, technical_db=TECHNICAL_QUESTIONS,
                                    coding=CODING_QUESTIONS, hr=HR_QUESTIONS, behavioral=BEHAVIORAL_QUESTIONS):
    technical_questions = []
    for skill in technical_skills:
        skill_lower = skill.lower()
        if skill_lower in technical_db:
            technical_questions.extend(technical_db[skill_lower])

    return {
        "technical_questions": list(dict.fromkeys(technical_questions)),
        "coding_questions": coding,
        "hr_questions": hr,
        "behavioral_questions": behavioral
    }

template_questions = generate_interview_questions(technical_skills)
print("Template Question Bank: SUCCESS")
print(f"Generated {len(template_questions['technical_questions'])} technical questions from templates")

Template Question Bank: SUCCESS
Generated 10 technical questions from templates


In [10]:
def get_llm_interview_questions(resume_text, target_role):
    prompt = f"""You are a senior technical interviewer hiring for an {target_role} position.

Based on THIS candidate's actual resume below, generate 5 interview questions that probe specifically into THEIR projects and experience (not generic questions).
Reference their actual project names or skills where relevant.

RESUME TEXT:
{resume_text[:2000]}

Respond as a numbered list of exactly 5 questions. No preamble, just the list."""

    return call_llm(prompt, temperature=0.6, max_tokens=350)

start = time.time()
llm_personalized_questions = get_llm_interview_questions(resume_text, target_role)
elapsed = time.time() - start

print(f"LLM Personalized Interview Questions: SUCCESS (generated in {elapsed:.2f}s)\n")
print(llm_personalized_questions)

LLM Personalized Interview Questions: SUCCESS (generated in 4.53s)

1. In your "Predictive Supply Chain AI" project, which specific machine learning algorithm did you select to model disruptions, and what data preprocessing steps using Pandas or NumPy were critical for ensuring the model's accuracy?
2. Can you walk me through how Flask was integrated into your supply chain system to serve predictions dynamically, specifically regarding API design endpoints required by external stakeholders?
3. Given that you built a responsive portfolio site from scratch using HTML, CSS, and JavaScript, what specific frontend optimization techniques did you employ to ensure fast load times for users on different devices?
4. Your resume highlights both C++/Java proficiency alongside Python; how do you plan to leverage these languages in your AI engineering workflow versus when would you strictly prefer using pure Python libraries like PyTorch or TensorFlow?
5. With an SGPA of 8.4 and experience leading 

In [11]:
combined_interview_set = {
    "template_questions": template_questions,
    "llm_personalized_questions": llm_personalized_questions,
    "llm_generation_time_seconds": round(elapsed, 2)
}

with open("../outputs/interview_questions.json", "w", encoding="utf-8") as f:
    json.dump(combined_interview_set, f, indent=2)

print("Combined interview questions (templates + LLM personalized) saved")
print("Notebook 9 (Interview Preparation Agent) — UPGRADED WITH LLM — COMPLETE")

Combined interview questions (templates + LLM personalized) saved
Notebook 9 (Interview Preparation Agent) — UPGRADED WITH LLM — COMPLETE
